# Imports

Import packages and set paths - largely as in the previous notebook

In [ ]:
%load_ext autoreload
%autoreload 2

from joblib import Parallel, delayed
from itertools import product
from tqdm import tqdm
import megcourselib
import mne
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from skimage.filters import gabor
import matplotlib.pyplot as plt
from skimage.filters import gabor_kernel
import numpy as np
from scipy.ndimage import convolve
from scipy.signal import fftconvolve

from megcourselib.data.util import get_scene_path
from megcourselib.data.stim import extract_crop

import os
from os import path

In [ ]:
PROJECT_ROOT = ".."
CHALLENGE = "challenge1"
DATA_ROOT = path.join(PROJECT_ROOT, 'data', 'brainencoding26', CHALLENGE)
STIM_ROOT = path.join(PROJECT_ROOT, 'data', 'avs_scenes')

# Feature Extraction - Gabor Wavelets

Below is an example for how to build and apply Gabor filters to a stimulus. This code is not very efficient, but should serve as an example for the general process of building a set of predictors from 'hand-crafted' features.

In the next notebook, we will follow the same general procedure for building a set of predictors - this time by extracting activations from a neural network. The general process for craeting an encoding model is always the same:

1. Formulate a hypothesis for a set of features from which you want to predict neural data
2. Build a set of predictors for all stimuli in your dataset
3. fit an encoding model

In this case, we build a set of Gabor filters to extract features from our images:

In [ ]:
frequencies = [0.1, 0.2, 0.4]  # watch out: lower frequencies result in larger kernels. This becomes expensive quite quickly.
thetas = [i * np.pi / 4 for i in range(8)]

fig, axes = plt.subplots(
    len(frequencies), len(thetas),
    figsize=(len(thetas) * 2.5, len(frequencies) * 2.5)
)

for i, freq in enumerate(frequencies):
    for j, theta in enumerate(thetas):
        k = gabor_kernel(freq, theta=theta)

        axes[i, j].imshow(k.real, cmap="RdBu_r", vmin=-k.real.max(), vmax=k.real.max())
        axes[i, j].set_title(f"f={freq}, θ={theta/np.pi:.2f}π", fontsize=8)
        axes[i, j].axis("off")

fig.suptitle("Gabor filter bank (real part)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# show a single kernel
freq, theta = 0.1, 3/4 * np.pi
k = gabor_kernel(freq, theta=theta)
plt.imshow(k.real, cmap='RdBu_r', vmin=-k.real.max(), vmax=k.real.max())
plt.axis('off')
plt.show()

## Load the Metadata and apply the filters to a single image

In [ ]:
# get metadata, as before
metadata = pd.read_csv(path.join(DATA_ROOT, 'training', 'metadata.csv'))

In [ ]:
# select a fixation, and show the image
IDX = 43

scene_id = metadata.sceneID[IDX]
image_path = get_scene_path(scene_id, STIM_ROOT)

im = Image.open(image_path).convert('RGB')
im_arr = np.asarray(im)

plt.figure(figsize=(8,6))
plt.imshow(im_arr)
plt.axis('off')
plt.title(f"Scene ID: {int(scene_id)}")
plt.show()

In [ ]:
# get a crop
gx = metadata.iloc[IDX].mean_gx
gy = metadata.iloc[IDX].mean_gy

crop, is_valid = extract_crop(im, gx, gy, crop_w=244, crop_h=244)
assert is_valid, 'invalid crop'

plt.figure()
plt.imshow(crop)
plt.axis('off')
plt.show()

In [ ]:
GRAYSCALE = np.array([0.2126, 0.7152, 0.0722])
crop_arr_gray = (crop * GRAYSCALE[None, None, :]).sum(axis=-1)

fig, axes = plt.subplots(
    len(frequencies), len(thetas),
    figsize=(len(thetas) * 2.5, len(frequencies) * 2.5)
)

for i, freq in enumerate(frequencies):
    for j, theta in enumerate(thetas):
        k = gabor_kernel(freq, theta=theta)
        real = convolve(crop_arr_gray, k.real)
        imag = convolve(crop_arr_gray, k.imag)
        energy = np.sqrt(real**2 + imag**2)

        axes[i, j].imshow(energy, cmap="inferno")
        axes[i, j].set_title(f"f={freq}, θ={theta/np.pi:.2f}π", fontsize=8)
        axes[i, j].axis("off")

fig.suptitle("Gabor filter bank — energy responses", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# show for a single image
freq, theta = 0.1, 3/4 * np.pi
k = gabor_kernel(freq, theta=theta)
real = convolve(crop_arr_gray, k.real)
imag = convolve(crop_arr_gray, k.imag)
energy = np.sqrt(real**2 + imag**2)

plt.figure()
plt.imshow(energy, cmap="inferno")
plt.axis("off")


# Questions for Discussion:

1. Do you expect this type of feature to be a good predictor for the MEG data?
2. For which sensors do you expect this type of feature to work best?
3. What kinds of hypotheses would you test with this type of predictor?
4. What follow up experiments can you think of? (think of design choices you made when creating your predictors)

# Creating your first set of predictors for an encoding model

Below is an example pipeline to extract the features for the data for one subject.
We will use it to train a first encoding model.

Note that this pipeline is still not very efficient (or reproducible) but is meant to give you an idea of the general process. In the next sessions we will improve on this.

In [ ]:
def build_filter_bank(frequencies, thetas):
    """Build a list of Gabor kernels once, reuse across images."""
    return [
        gabor_kernel(freq, theta=theta)
        for freq, theta in product(frequencies, thetas)
    ]

def extract_features(image, filter_bank, grid=(4, 4)):
    """Mean energy per grid cell per filter. Shape: (n_filters * grid_h * grid_w,)."""
    GRAYSCALE = np.array([0.2126, 0.7152, 0.0722])
    image = (image * GRAYSCALE[None, None, :]).sum(axis=-1)
    gh, gw = grid
    h, w   = image.shape
    # make sure the image size is a multiple of the grid size
    image  = image[:h - h % gh, :w - w % gw]
    features = []
    for k in filter_bank:
        real   = fftconvolve(image, k.real, mode='same')
        imag   = fftconvolve(image, k.imag, mode='same')
        energy = np.sqrt(real**2 + imag**2)
        tiled  = energy.reshape(gh, energy.shape[0] // gh, gw, energy.shape[1] // gw)
        features.append(tiled.mean(axis=(1, 3)).ravel())
    return np.concatenate(features)

def get_features_for_fixation(scene_id, gx, gy, cropsize, filter_bank, grid):
    im = Image.open(get_scene_path(scene_id, STIM_ROOT)).convert('RGB')
    crop, is_valid = extract_crop(im, gx, gy, crop_w=cropsize[0], crop_h=cropsize[1])
    if not is_valid:
        feats = np.empty(len(filter_bank) * grid[0] * grid[1])
        feats.fill(np.nan)
    else:
        crop = crop / 255.  # convert to float
        feats = extract_features(np.array(crop), filter_bank, grid=grid)
    return feats

def process_row(row, cropsize, filter_bank, grid):
    return get_features_for_fixation(row.sceneID, row.mean_gx, row.mean_gy, cropsize, filter_bank, grid)


In [ ]:
frequencies = [0.1, 0.4]
thetas = [i * np.pi / 4 for i in range(4)]
grid = (8, 8)
subject = 1
cropsize = (244, 244)

subj_metadata = metadata[metadata.subject == subject]
filter_bank = build_filter_bank(frequencies, thetas)

nrows = len(subj_metadata)
nfeat = len(filter_bank) * grid[0] * grid[1]

predictors = np.empty((nrows, nfeat))

with Parallel(n_jobs=-1, return_as="generator") as parallel:
    results = parallel(
        delayed(process_row)(row, cropsize, filter_bank, grid)
        for _, row in subj_metadata.iterrows()
    )
    for i, feats in enumerate(tqdm(results, total=nrows)):
        predictors[i] = feats

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler

# load meg data
meg_train = np.load(path.join(DATA_ROOT, 'training', 'meg_110ms.npy'))

# get data to match the indices for which we just extracted features
meg = meg_train[subj_metadata.index]

# predictors contain nan if the crop is out of bounds. Count how often this happens and apply a mask
has_nan = np.isnan(predictors).any(axis=1)
print(f"{has_nan.sum()}/{len(predictors)} crops are invalid and will be excluded.")

# get final data for encoding
meg = meg[~has_nan]
pred = predictors[~has_nan]
meta = subj_metadata.loc[~has_nan].reset_index()

# standardize the meg data
scaler_meg = RobustScaler().fit(meg)
meg = scaler_meg.transform(meg)
meg  = scaler_meg.transform(meg)

print()
print()
print('========================== DATA FOR ENCODING MODEL ==========================')
print('MEG data:', meg.shape, 'predictors:', pred.shape, 'metadata:', meta.shape)


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import explained_variance_score
import numpy as np

def correlation_score(y_true, y_pred):
    return np.array([
        np.corrcoef(y_true[:, c], y_pred[:, c])[0, 1]
        for c in range(y_true.shape[1])
    ])

nfold = 5
alphas = np.logspace(-3, 16, 128)
kf = KFold(n_splits=nfold, shuffle=True, random_state=42)
n_trials, n_chan = meg.shape

r_per_fold  = np.empty((nfold, n_chan))



for i, (train_idx, test_idx) in enumerate(kf.split(pred)):
    print(f">>> fold {i + 1} / {nfold}")

    X_train, X_test = pred[train_idx], pred[test_idx]
    y_train, y_test = meg[train_idx], meg[test_idx]

    scaler_pred = StandardScaler().fit(X_train)
    X_train = scaler_pred.transform(X_train)
    X_test  = scaler_pred.transform(X_test)


    model = RidgeCV(alphas=alphas, alpha_per_target=True).fit(X_train, y_train)
    predictions = model.predict(X_test)

    r_per_fold[i]  = correlation_score(y_test, predictions)

print(f"r:  mean={r_per_fold.mean():.5f}, worst sensor={r_per_fold.mean(axis=0).min():.5f}  best sensor={r_per_fold.mean(axis=0).max():.5f}")


In [ ]:
# show on topoplot

# get information about sensor location for the subject
fif_path = path.join(DATA_ROOT, 'training', f"sub-{str(subject).zfill(2)}_grad_info.fif")
info = mne.io.read_info(fif_path)

# plot data
fig, ax = plt.subplots(figsize=(6, 6))
im, _ = mne.viz.plot_topomap(
    data=r_per_fold.mean(axis=0),
    pos=info,
    ch_type='grad',
    axes=ax,
    show=False,          # prevent auto-display so we can add the colorbar
    vlim=(r_per_fold.mean(axis=0).min(), r_per_fold.mean(axis=0).max())
)

plt.colorbar(im, ax=ax, shrink=0.7, label='pearson correlation')
plt.tight_layout()
plt.show()


# Next steps

Before moving on to the next notebook, feel free to experiment with the code above. What happens when you change the settings for feature extraction? You can also try to use the encoding model above and apply it to the data from one of the other participants. Does the model generalize?